In [0]:

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

print("Libraries imported successfully.")
print(f"Spark version: {spark.version}")

Libraries imported successfully.
Spark version: 4.2.0


In [0]:

bronze_df = spark.table("healthcare_readmission.readmission.bronze_diabetic_data")

print("Bronze table loaded successfully.")
print(f"Table: healthcare_readmission.readmission.bronze_diabetic_data")

Bronze table loaded successfully.
Table: healthcare_readmission.readmission.bronze_diabetic_data


In [0]:

print("=== BRONZE TABLE SCHEMA ===")
bronze_df.printSchema()

print("\n=== SAMPLE DATA (first 5 rows) ===")
display(bronze_df.limit(5))

=== BRONZE TABLE SCHEMA ===
root
 |-- encounter_id: integer (nullable = true)
 |-- patient_nbr: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- number_outpatient: integer (nullable = true)
 |-- number_emergency: integer (nullable = true)
 |-- number_inpatient: integer (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: i

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [0]:
num_rows = bronze_df.count()
num_cols = len(bronze_df.columns)

print(f"Total rows:    {num_rows:,}")
print(f"Total columns: {num_cols}")
print(f"\nColumn names: {bronze_df.columns}")

Total rows:    101,766
Total columns: 50

Column names: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when, count, lit

# Build a list of (column_name, null_count, question_mark_count) tuples
null_qm_data = []

total_rows = bronze_df.count()

for c in bronze_df.columns:
    null_count = bronze_df.filter(col(c).isNull()).count()
    # For string columns, count '?' placeholders; for numeric, always 0
    if dict(bronze_df.dtypes)[c] == 'string':
        qm_count = bronze_df.filter(col(c) == '?').count()
    else:
        qm_count = 0
    null_qm_data.append((c, null_count, qm_count))

# Create a summary DataFrame for readable display
null_qm_df = spark.createDataFrame(
    null_qm_data,
    schema=["column_name", "null_count", "question_mark_count"]
)

# Add percentage columns for easier interpretation
null_qm_df = null_qm_df.withColumn("null_pct", F.round((col("null_count") / lit(total_rows)) * 100, 2)) \
    .withColumn("qm_pct", F.round((col("question_mark_count") / lit(total_rows)) * 100, 2)) \
    .orderBy(col("question_mark_count").desc())

print(f"=== NULL AND '?' PLACEHOLDER ANALYSIS (total rows: {total_rows:,}) ===")
display(null_qm_df)

# Print a quick text summary of columns with '?' values
print("\n=== COLUMNS WITH '?' PLACEHOLDER VALUES ===")
for row in null_qm_data:
    if row[2] > 0:
        pct = round((row[2] / total_rows) * 100, 2)
        print(f"  {row[0]:<25s}  '?' count: {row[2]:>7,}  ({pct}%)")

=== NULL AND '?' PLACEHOLDER ANALYSIS (total rows: 101,766) ===


column_name,null_count,question_mark_count,null_pct,qm_pct
weight,0,98569,0.0,96.86
medical_specialty,0,49949,0.0,49.08
payer_code,0,40256,0.0,39.56
race,0,2273,0.0,2.23
diag_3,0,1423,0.0,1.4
diag_2,0,358,0.0,0.35
diag_1,0,21,0.0,0.02
encounter_id,0,0,0.0,0.0
patient_nbr,0,0,0.0,0.0
gender,0,0,0.0,0.0



=== COLUMNS WITH '?' PLACEHOLDER VALUES ===
  race                       '?' count:   2,273  (2.23%)
  weight                     '?' count:  98,569  (96.86%)
  payer_code                 '?' count:  40,256  (39.56%)
  medical_specialty          '?' count:  49,949  (49.08%)
  diag_1                     '?' count:      21  (0.02%)
  diag_2                     '?' count:     358  (0.35%)
  diag_3                     '?' count:   1,423  (1.4%)


In [0]:
total_rows = bronze_df.count()
distinct_rows = bronze_df.distinct().count()
duplicate_count = total_rows - distinct_rows

print(f"Total rows:        {total_rows:,}")
print(f"Distinct rows:     {distinct_rows:,}")
print(f"Duplicate rows:    {duplicate_count:,}")

if duplicate_count > 0:
    print(f"\nDuplicates found! {duplicate_count:,} rows are exact duplicates.")
else:
    print("\nNo exact duplicate rows found.")

Total rows:        101,766
Distinct rows:     101,766
Duplicate rows:    0

No exact duplicate rows found.


In [0]:
total_rows = bronze_df.count()

target_dist = bronze_df.groupBy("readmitted") \
    .count() \
    .withColumn("percentage", F.round((col("count") / lit(total_rows)) * 100, 2)) \
    .orderBy(col("count").desc())

print("=== TARGET DISTRIBUTION: readmitted ===")
display(target_dist)

# Also print as text for clarity
print("\nText summary:")
for row in target_dist.collect():
    print(f"  {row['readmitted']:<5s}  count: {row['count']:>7,}  ({row['percentage']}%)")

=== TARGET DISTRIBUTION: readmitted ===


readmitted,count,percentage
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16



Text summary:
  NO     count:  54,864  (53.91%)
  >30    count:  35,545  (34.93%)
  <30    count:  11,357  (11.16%)


In [0]:
# ============================================================
# Cell 8: Identify Identifier Columns and Non-Feature Columns
# ============================================================
# Some columns are identifiers or have zero variance and should
# NOT be used as ML features. We identify them here but do NOT
# remove them yet — removal happens later after all cleaning.

# --- Identifier columns (unique per row or per patient) ---
encounter_id_distinct = bronze_df.select("encounter_id").distinct().count()
patient_nbr_distinct = bronze_df.select("patient_nbr").distinct().count()
total_rows = bronze_df.count()

print("=== IDENTIFIER COLUMNS ANALYSIS ===")
print(f"encounter_id:  {encounter_id_distinct:,} distinct values out of {total_rows:,} rows")
print(f"  -> Every encounter has a unique ID. This is a pure row identifier.")
print(f"     Must be EXCLUDED from ML features (no predictive value, risk of leakage).")
print()
print(f"patient_nbr:   {patient_nbr_distinct:,} distinct values out of {total_rows:,} rows")
print(f"  -> Some patients have multiple encounters. patient_nbr identifies")
print(f"     the person, not the visit. Including it risks data leakage because")
print(f"     a model could memorise patient-specific readmission patterns.")
print(f"     Must be EXCLUDED from ML features.")
print()

# --- Zero-variance columns (only one unique value) ---
print("=== ZERO-VARIANCE COLUMN CHECK ===")
for c in bronze_df.columns:
    if dict(bronze_df.dtypes)[c] == 'string':
        n_distinct = bronze_df.select(c).distinct().count()
        if n_distinct == 1:
            only_val = bronze_df.select(c).distinct().first()[c]
            print(f"  {c:<25s}  unique values: {n_distinct}  (only '{only_val}')")
            print(f"    -> This column has zero variance and adds no information for ML.")
            print(f"       It will be kept in Silver but noted for the Gold notebook.")
print()
print("Columns that will be EXCLUDED from the Silver ML-ready dataset:")
print("  - encounter_id  (pure identifier, zero predictive value)")
print("  - patient_nbr   (patient identifier, data leakage risk)")
print("All other columns are retained as they may carry clinical value.")

=== IDENTIFIER COLUMNS ANALYSIS ===
encounter_id:  101,766 distinct values out of 101,766 rows
  -> Every encounter has a unique ID. This is a pure row identifier.
     Must be EXCLUDED from ML features (no predictive value, risk of leakage).

patient_nbr:   71,518 distinct values out of 101,766 rows
  -> Some patients have multiple encounters. patient_nbr identifies
     the person, not the visit. Including it risks data leakage because
     a model could memorise patient-specific readmission patterns.
     Must be EXCLUDED from ML features.

=== ZERO-VARIANCE COLUMN CHECK ===
  examide                    unique values: 1  (only 'No')
    -> This column has zero variance and adds no information for ML.
       It will be kept in Silver but noted for the Gold notebook.
  citoglipton                unique values: 1  (only 'No')
    -> This column has zero variance and adds no information for ML.
       It will be kept in Silver but noted for the Gold notebook.

Columns that will be EXCLU

In [0]:
# ============================================================
# Cell 9: Handle '?' Placeholder Values
# ============================================================
# The dataset uses '?' as a placeholder for missing data in
# several string columns. We replace '?' with 'Unknown' for
# categorical columns where the value is genuinely missing.
#
# Note: 'None' in max_glu_serum and A1Cresult is NOT a missing
# placeholder — it means the test was not performed. We keep it.
#
# Columns with '?' values (from our analysis):
#   race              (~2.2%  missing)
#   weight            (~96.9% missing)
#   payer_code        (~39.6% missing)
#   medical_specialty (~49.1% missing)
#   diag_1            (~0.02% missing)
#   diag_2            (~0.35% missing)
#   diag_3            (~1.4%  missing)

# Work on a copy so the Bronze DataFrame stays untouched
cleaned_df = bronze_df

# List of string columns where '?' should be replaced with 'Unknown'
question_mark_cols = [
    "race", "gender", "weight", "payer_code", "medical_specialty",
    "diag_1", "diag_2", "diag_3"
]

for c in question_mark_cols:
    cleaned_df = cleaned_df.withColumn(
        c,
        F.when(col(c) == "?", "Unknown").otherwise(col(c))
    )

print("'?' placeholder values replaced with 'Unknown' in columns:")
for c in question_mark_cols:
    print(f"  - {c}")

# Verify no '?' remains in any string column
remaining_qm = 0
for c in cleaned_df.columns:
    if dict(cleaned_df.dtypes)[c] == 'string':
        cnt = cleaned_df.filter(col(c) == "?").count()
        if cnt > 0:
            print(f"  WARNING: '{c}' still has {cnt} '?' values!")
            remaining_qm += cnt

if remaining_qm == 0:
    print("\nVerification passed: no '?' values remain in any column.")

'?' placeholder values replaced with 'Unknown' in columns:
  - race
  - gender
  - weight
  - payer_code
  - medical_specialty
  - diag_1
  - diag_2
  - diag_3

Verification passed: no '?' values remain in any column.


In [0]:
# ============================================================
# Cell 10: Handle Remaining Missing Values
# ============================================================
# After replacing '?' with 'Unknown', check for any remaining
# nulls. Numeric columns in this dataset do not contain '?' or
# nulls (they were loaded as INT from the Bronze layer), so no
# numerical imputation is needed.
#
# For the 'gender' column, also check for 'Unknown/Invalid' which
# is a rare but known invalid value in this dataset.

# Check for any true NULLs across all columns
print("=== REMAINING NULL CHECK ===")
any_nulls = False
for c in cleaned_df.columns:
    null_cnt = cleaned_df.filter(col(c).isNull()).count()
    if null_cnt > 0:
        print(f"  {c}: {null_cnt} nulls")
        any_nulls = True

if not any_nulls:
    print("  No null values found in any column.")

# Check gender for 'Unknown/Invalid' values
gender_values = cleaned_df.groupBy("gender").count().orderBy(col("count").desc())
print("\n=== GENDER VALUE DISTRIBUTION ===")
display(gender_values)

# Replace 'Unknown/Invalid' in gender with 'Unknown' for consistency
cleaned_df = cleaned_df.withColumn(
    "gender",
    F.when(col("gender") == "Unknown/Invalid", "Unknown").otherwise(col("gender"))
)

print("\nGender 'Unknown/Invalid' values (if any) replaced with 'Unknown'.")
print("No numerical imputation needed — all INT columns are complete.")

=== REMAINING NULL CHECK ===
  No null values found in any column.

=== GENDER VALUE DISTRIBUTION ===


gender,count
Female,54708
Male,47055
Unknown/Invalid,3



Gender 'Unknown/Invalid' values (if any) replaced with 'Unknown'.
No numerical imputation needed — all INT columns are complete.


In [0]:
# ============================================================
# Cell 11: Remove Exact Duplicate Records
# ============================================================
# If any exact duplicate rows exist (all columns identical),
# we remove them to prevent biased model training.

before_count = cleaned_df.count()
cleaned_df = cleaned_df.dropDuplicates()
after_count = cleaned_df.count()
removed = before_count - after_count

print(f"Rows before dedup:  {before_count:,}")
print(f"Rows after dedup:   {after_count:,}")
print(f"Duplicates removed: {removed:,}")

if removed == 0:
    print("No exact duplicate rows found — nothing removed.")

Rows before dedup:  101,766
Rows after dedup:   101,766
Duplicates removed: 0
No exact duplicate rows found — nothing removed.


In [0]:
# ============================================================
# Cell 12: Convert Columns to Suitable Data Types
# ============================================================
# The 'age' column is stored as string ranges like '[0-10)',
# '[10-20)', etc. We convert it to an ordinal integer (0-9)
# so it can be used numerically by ML models later.
# All other columns already have appropriate types from Bronze.

# Define the mapping from age range string to ordinal integer
age_mapping = {
    "[0-10)":   0,
    "[10-20)":  1,
    "[20-30)":  2,
    "[30-40)":  3,
    "[40-50)":  4,
    "[50-60)":  5,
    "[60-70)":  6,
    "[70-80)":  7,
    "[80-90)":  8,
    "[90-100)": 9
}

# Build a CASE expression for the mapping
age_expr = F.lit(None).cast("int")
for age_range, ordinal in age_mapping.items():
    age_expr = F.when(col("age") == age_range, F.lit(ordinal)).otherwise(age_expr)

cleaned_df = cleaned_df.withColumn("age", age_expr.cast("int"))

print("Converted 'age' from string ranges to ordinal integers (0-9).")
print("Example mapping: '[0-10)' -> 0, '[10-20)' -> 1, ..., '[90-100)' -> 9")

# Verify the conversion
print("\n=== AGE COLUMN AFTER CONVERSION ===")
display(cleaned_df.groupBy("age").count().orderBy("age"))

Converted 'age' from string ranges to ordinal integers (0-9).
Example mapping: '[0-10)' -> 0, '[10-20)' -> 1, ..., '[90-100)' -> 9

=== AGE COLUMN AFTER CONVERSION ===


age,count
0,161
1,691
2,1657
3,3775
4,9685
5,17256
6,22483
7,26068
8,17197
9,2793


In [0]:
# ============================================================
# Cell 13: Clean the 'readmitted' Target Column
# ============================================================
# The target column 'readmitted' already has three valid
# categories: 'NO', '>30', '<30'. We verify that no invalid or
# unexpected values exist and trim any whitespace.
#
# We preserve all three categories as-is because:
#   - 'NO'  = patient was not readmitted
#   - '>30' = readmitted after more than 30 days
#   - '<30' = readmitted within 30 days
# These distinctions are clinically meaningful and the Gold/ML
# notebook will decide whether to binarise or keep as multi-class.

# Trim whitespace and verify values
cleaned_df = cleaned_df.withColumn("readmitted", F.trim(col("readmitted")))

target_values = cleaned_df.groupBy("readmitted").count().orderBy(col("count").desc())
print("=== READMITTED TARGET (after cleaning) ===")
display(target_values)

# Verify only the three expected values exist
valid_values = {"NO", ">30", "<30"}
actual_values = set(row["readmitted"] for row in target_values.collect())

if actual_values == valid_values:
    print(f"\nTarget is clean: only {valid_values} present.")
else:
    print(f"\nWARNING: unexpected values found: {actual_values - valid_values}")

=== READMITTED TARGET (after cleaning) ===


readmitted,count
NO,54864
>30,35545
<30,11357



Target is clean: only {'>30', 'NO', '<30'} present.


In [0]:
# ============================================================
# Cell 14: Remove Identifier Columns from Silver Dataset
# ============================================================
# We remove only the two identifier columns that have no
# predictive value and pose a data-leakage risk:
#
#   - encounter_id: unique per encounter (pure row identifier)
#   - patient_nbr:  unique per patient (could leak patient-level
#                  readmission history into the model)
#
# All other columns are retained, including zero-variance columns
# (examide, citoglipton) — the Gold/ML notebook can decide whether
# to drop them during feature selection.

cols_to_drop = ["encounter_id", "patient_nbr"]

print(f"Columns being removed: {cols_to_drop}")
print("Reason: these are identifiers with no clinical predictive value.")
print("Removing patient_nbr prevents data leakage from patient-level patterns.")
print("\nAll clinical, medication, and admission columns are retained.")

cleaned_df = cleaned_df.drop(*cols_to_drop)

print(f"\nColumns after removal: {len(cleaned_df.columns)}")
print(f"Remaining columns: {cleaned_df.columns}")

Columns being removed: ['encounter_id', 'patient_nbr']
Reason: these are identifiers with no clinical predictive value.
Removing patient_nbr prevents data leakage from patient-level patterns.

All clinical, medication, and admission columns are retained.

Columns after removal: 48
Remaining columns: ['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'met

In [0]:
# ============================================================
# Cell 15: Final Data-Quality Check After Cleaning
# ============================================================
# We re-run the same null and '?' checks on the cleaned dataset
# to confirm everything is in order before saving.

print("=== FINAL DATA-QUALITY CHECK ===")
print(f"Total rows:    {cleaned_df.count():,}")
print(f"Total columns: {len(cleaned_df.columns)}")

# 1. Check for any remaining '?' values
qm_found = False
for c in cleaned_df.columns:
    if dict(cleaned_df.dtypes)[c] == 'string':
        cnt = cleaned_df.filter(col(c) == "?").count()
        if cnt > 0:
            print(f"  WARNING: '{c}' still has {cnt} '?' values!")
            qm_found = True
if not qm_found:
    print("1. No '?' placeholder values remain. ✓")

# 2. Check for any NULL values
null_found = False
for c in cleaned_df.columns:
    cnt = cleaned_df.filter(col(c).isNull()).count()
    if cnt > 0:
        print(f"  WARNING: '{c}' has {cnt} NULL values!")
        null_found = True
if not null_found:
    print("2. No NULL values found. ✓")

# 3. Check for duplicates
dup_count = cleaned_df.count() - cleaned_df.distinct().count()
if dup_count == 0:
    print("3. No duplicate rows. ✓")
else:
    print(f"3. WARNING: {dup_count} duplicate rows still exist!")

# 4. Check identifier columns are removed
if "encounter_id" not in cleaned_df.columns and "patient_nbr" not in cleaned_df.columns:
    print("4. Identifier columns (encounter_id, patient_nbr) removed. ✓")
else:
    print("4. WARNING: identifier columns still present!")

# 5. Check target column has only valid values
target_vals = set(row["readmitted"] for row in cleaned_df.select("readmitted").distinct().collect())
if target_vals == {"NO", ">30", "<30"}:
    print(f"5. Target 'readmitted' has only valid values: {target_vals}. ✓")
else:
    print(f"5. WARNING: unexpected target values: {target_vals}")

print("\nData-quality check complete.")

=== FINAL DATA-QUALITY CHECK ===
Total rows:    101,766
Total columns: 48
1. No '?' placeholder values remain. ✓
2. No NULL values found. ✓
3. No duplicate rows. ✓
4. Identifier columns (encounter_id, patient_nbr) removed. ✓
5. Target 'readmitted' has only valid values: {'>30', 'NO', '<30'}. ✓

Data-quality check complete.


In [0]:
# ============================================================
# Cell 16: Display Cleaned Dataset and Schema
# ============================================================

print("=== CLEANED SILVER DATASET SCHEMA ===")
cleaned_df.printSchema()

print("\n=== SAMPLE OF CLEANED DATA (first 10 rows) ===")
display(cleaned_df.limit(10))

=== CLEANED SILVER DATASET SCHEMA ===
root
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- number_outpatient: integer (nullable = true)
 |-- number_emergency: integer (nullable = true)
 |-- number_inpatient: integer (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: integer (nullable = true)
 |-- max_glu_serum: string (nullable = true)
 |-- A1C

race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
Caucasian,Female,0,Unknown,6,25,1,1,Unknown,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,Unknown,Unknown,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
Caucasian,Female,1,Unknown,1,1,7,3,Unknown,Unknown,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
AfricanAmerican,Female,2,Unknown,1,1,7,2,Unknown,Unknown,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
Caucasian,Male,3,Unknown,1,1,7,2,Unknown,Unknown,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
Caucasian,Male,4,Unknown,1,1,7,1,Unknown,Unknown,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
Caucasian,Male,5,Unknown,2,1,2,3,Unknown,Unknown,31,6,16,0,0,0,414,411,250,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
Caucasian,Male,6,Unknown,3,1,2,4,Unknown,Unknown,70,1,21,0,0,0,414,411,V45,7,None,None,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
Caucasian,Male,7,Unknown,1,1,7,5,Unknown,Unknown,73,0,12,0,0,0,428,492,250,8,None,None,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
Caucasian,Female,8,Unknown,2,1,4,13,Unknown,Unknown,68,2,28,0,0,0,398,427,38,8,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
Caucasian,Female,9,Unknown,3,3,4,12,Unknown,InternalMedicine,33,3,18,0,0,0,434,198,486,8,None,None,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [0]:
# ============================================================
# Cell 17: Final Row Count and Target Distribution
# ============================================================

final_rows = cleaned_df.count()
final_cols = len(cleaned_df.columns)

print(f"Final Silver dataset:")
print(f"  Rows:    {final_rows:,}")
print(f"  Columns: {final_cols}")

print("\n=== FINAL TARGET DISTRIBUTION ===")
final_target = cleaned_df.groupBy("readmitted") \
    .count() \
    .withColumn("percentage", F.round((col("count") / lit(final_rows)) * 100, 2)) \
    .orderBy(col("count").desc())

display(final_target)

print("\nText summary:")
for row in final_target.collect():
    print(f"  {row['readmitted']:<5s}  count: {row['count']:>7,}  ({row['percentage']}%)")

Final Silver dataset:
  Rows:    101,766
  Columns: 48

=== FINAL TARGET DISTRIBUTION ===


readmitted,count,percentage
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16



Text summary:
  NO     count:  54,864  (53.91%)
  >30    count:  35,545  (34.93%)
  <30    count:  11,357  (11.16%)


In [0]:
# ============================================================
# Cell 18: Save Cleaned Data as Silver Delta Table
# ============================================================
# Write the cleaned DataFrame to:
#   healthcare_readmission.readmission.silver_patient_data
# We use mode 'overwrite' to create/replace the Silver table.
# This does NOT modify the Bronze source table in any way.

TARGET_TABLE = "healthcare_readmission.readmission.silver_patient_data"

print(f"Writing Silver Delta table: {TARGET_TABLE}")

cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TABLE)

print(f"\nSilver table written successfully: {TARGET_TABLE}")

Writing Silver Delta table: healthcare_readmission.readmission.silver_patient_data

Silver table written successfully: healthcare_readmission.readmission.silver_patient_data


In [0]:
# ============================================================
# Cell 19: Verify the Silver Delta Table Was Created
# ============================================================
# Read the table back from Unity Catalog and confirm it matches
# the cleaned DataFrame we just wrote.

TARGET_TABLE = "healthcare_readmission.readmission.silver_patient_data"

# Read the table back
silver_table = spark.table(TARGET_TABLE)

verify_rows = silver_table.count()
verify_cols = len(silver_table.columns)

print(f"=== SILVER TABLE VERIFICATION ===")
print(f"Table:     {TARGET_TABLE}")
print(f"Rows:      {verify_rows:,}")
print(f"Columns:  {verify_cols}")
print(f"\nColumn names: {silver_table.columns}")

# Confirm row count matches the cleaned DataFrame
if verify_rows == cleaned_df.count():
    print(f"\n✓ Row count matches cleaned DataFrame ({verify_rows:,} rows).")
else:
    print(f"\n✗ Row count MISMATCH: table={verify_rows:,} vs cleaned_df={cleaned_df.count():,}")

# Confirm no '?' or NULL values in the saved table
qm_total = 0
null_total = 0
for c in silver_table.columns:
    if dict(silver_table.dtypes)[c] == 'string':
        qm_total += silver_table.filter(col(c) == "?").count()
    null_total += silver_table.filter(col(c).isNull()).count()

if qm_total == 0:
    print("✓ No '?' values found in Silver table.")
else:
    print(f"✗ {qm_total} '?' values found in Silver table!")

if null_total == 0:
    print("✓ No NULL values found in Silver table.")
else:
    print(f"✗ {null_total} NULL values found in Silver table!")

print("\n=== SILVER TABLE SAMPLE (first 5 rows) ===")
display(silver_table.limit(5))

=== SILVER TABLE VERIFICATION ===
Table:     healthcare_readmission.readmission.silver_patient_data
Rows:      101,766
Columns:  48

Column names: ['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

✓ Row count matches cleaned DataFrame (101,766 rows).
✓ No '?' va

race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
AfricanAmerican,Female,4,Unknown,1,3,7,3,Unknown,Unknown,64,0,15,0,0,1,428,250.03,707,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
Caucasian,Female,7,Unknown,3,1,1,14,Unknown,Surgery-General,28,3,28,0,0,0,151,196,553,8,None,None,Steady,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,NO
AfricanAmerican,Male,5,Unknown,1,6,7,3,Unknown,Unknown,52,0,13,0,0,2,198,197,197,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
AfricanAmerican,Female,5,Unknown,2,1,2,1,Unknown,Unknown,34,1,6,0,0,0,997,577,401,4,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
Caucasian,Male,2,Unknown,2,1,2,2,Unknown,Unknown,35,0,12,0,0,1,277,250.02,753,3,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
